In [63]:
import nltk

In [64]:
import numpy as np
import json
import glob
import string

In [65]:
#Gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

In [66]:
#vis
import pyLDAvis
import pyLDAvis.gensim_models

#spacy
import spacy

In [67]:
file_path1 = 'C:/Users/Pc/Documents/Unicamp/IC/2024/education_search_1.ris'
file_path2 = 'C:/Users/Pc/Documents/Unicamp/IC/2024/education_search_2.ris'

def criaArray(path):
    with open(path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    in_abs = False
    abs = []
    count = 0
    
    for line in lines:
        line = line.strip()
        if line.startswith('AB  - '):
            in_abs = True
            abs.append(line[6:])
    
    return abs

abstracts1 = criaArray(file_path1)
abstracts2 = criaArray(file_path2)

abstracts = []
abstracts.extend(abstracts1)
abstracts.extend(abstracts2)

array_unidimensional = np.array(abstracts)

# Transformando em um array bidimensional onde cada elemento seja um array de uma string
array_bidimensional = [[item] for item in array_unidimensional]

primeiros_abs = array_bidimensional[0:2]
# print(primeiros_abs)

In [68]:
def gen_words(texts):
    final = []
    for array in texts:
        for text in array:
            # Acessa o texto dentro de cada array e aplica simple_preprocess
            new = simple_preprocess(str(text), deacc=True)  # Convertendo para string aqui
            final.append(new)
    return final

data_words = gen_words(primeiros_abs)
# print(data_words)

In [69]:
print("quantidades de artigos: ", len(data_words))
print("quantidades de tokens artigo 1: ", len(data_words[0]))
print("quantidades de tokens artigo 1: ", len(data_words[1]))

quantidades de artigos:  2
quantidades de tokens artigo 1:  234
quantidades de tokens artigo 1:  209


In [70]:
from nltk.corpus import stopwords
stop_words = stopwords.words("english")

def filter_stopwords(words_list):
    filtered_result = []
    
    for sublist in words_list:
        filtered_sublist = []
        for word in sublist:
            if word.lower() not in stop_words:
                filtered_sublist.append(word)
        filtered_result.append(filtered_sublist)
    
    return filtered_result

data_words = filter_stopwords(data_words)
# print(data_words)

In [71]:
def lemmatization(texts, allowed_postags=["NOUN", "ADJ", "VERB", "ADV"]):
    #spacy é um modelo de processamento de linguagem natural. Aqui será usado para o método ".lemma_"
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    output = []
    
    for document in texts:
        lemma_abs = []  # Redefinir lemma_abs a cada iteração
        for word in document:
            doc = nlp(word)
            for token in doc:
                if token.pos_ in allowed_postags:
                    lemma_abs.append(token.lemma_)
    
        output.append(lemma_abs)
    return (output)

data_words = lemmatization(data_words)
# print(data_words)

In [72]:
# corpora.Dictionary elimina palavras repitidas
dictionary = corpora.Dictionary(data_words)
# doc2bow é um método do Dictionary que converte uma lista em  BoW
corpus = [dictionary.doc2bow(doc) for doc in data_words]

# (ID da palavra no dicionário, frequência da palavra no documento)
# print(corpus)

In [78]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                           id2word=dictionary,
                                           num_topics=10,
                                           random_state=100, #semente
                                           update_every=1, #frequência que o modelo é atualizado ao ver cada documento
                                           chunksize=50, #número de documentos a serem usados em cada iteração
                                           passes=10, #número de vezes que o modelo percorrerá o corpus inteiro durante o treinamento
                                           alpha="auto" #distribuição de tópicos por documento
                                           )

In [81]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, dictionary, mds="mmds", R=10)
vis

C:\Users\Pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\pyLDAvis\_prepare.py:247: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  by='saliency', ascending=False).head(R).drop('saliency', 1)


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
9     -0.104387  0.142446       1        1  53.919099
0      0.099186 -0.135587       2        1  45.932919
4      0.000326 -0.000295       3        1   0.020279
5      0.000589 -0.000937       4        1   0.020131
6      0.000483 -0.000700       5        1   0.019883
3      0.000862 -0.001070       6        1   0.019815
1      0.000700 -0.000840       7        1   0.019780
8      0.000711 -0.001037       8        1   0.016031
2      0.000781 -0.000978       9        1   0.016031
7      0.000750 -0.001003      10        1   0.016031, topic_info=              Term       Freq      Total Category  logprob  loglift
67         student  14.000000  14.000000  Default  10.0000  10.0000
42           learn  10.000000  10.000000  Default   9.0000   9.0000
52          online   6.000000   6.000000  Default   8.0000   8.0000
113           game   7.000000   7.000000  Default   7.0000   7.0000
24     educational   7.000000   7.000000  Default   6.0000   6.0000
..             ...        ...        ...      ...      ...      ...
0         academic   0.000277   1.985804  Topic10  -5.0239  -0.1378
8           author   0.000277   1.970295  Topic10  -5.0239  -0.1299
10   collaborative   0.000277   2.886723  Topic10  -5.0239  -0.5118
21          direct   0.000277   1.984723  Topic10  -5.0239  -0.1372
23       education   0.000277   1.984988  Topic10  -5.0239  -0.1373

[222 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
0         1  1.007149     academic
1         1  0.922534   additional
82        2  0.935934       amount
4         1  0.922741  application
85        2  0.935911        apply
...     ...       ...          ...
76        1  0.923444        usage
77        1  1.065937          use
150       2  1.022859         user
79        1  0.923047      various
151       2  1.083611        video

[79 rows x 3 columns], R=10, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[10, 1, 5, 6, 7, 4, 2, 9, 3, 8])